# UZK: готовый автоэнкодер → ансамбль GRU → Ridge

Запускайте из каталога проекта или ноутбука с ядром `.venv` после `uv sync`.
Сначала обучите и экспортируйте энкодер в `notebooks/uzk/gru_autoencoder/pretrain.ipynb`.
Конфиг `configs/uzk/autoencoder/ensemble.yaml` загружает `runs/uzk_gru_encoder.pkl`.

Сохранённый пайплайн включает scaler признаков, исключения, окна и энкодер.
Они не переобучаются. GRU обучаются на латентных признаках с известной целью,
Ridge — на их прогнозах на всей validation. Static selector здесь не используется.
Test вычисляется явно после обучения и загрузки сохранённого ансамбля.
Legacy-фильтр y использует всю переданную сетку, включая будущие цели и test;
это согласованное свойство исходного алгоритма.
Имена и порядок входных признаков и частота должны совпадать с экспортом;
допустимость периода предобучения проверяет автор эксперимента.


In [ ]:
import os
from pathlib import Path

# Resolve notebook paths from the project root, including nested notebook directories.
working_dir = Path.cwd().resolve()
project_root = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "pyproject.toml").is_file()
        and (directory / "src/feature_selector").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Open the notebook from within the feature_selector project")
os.chdir(project_root)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from feature_selector import load_config, run_experiment
from feature_selector.artifacts import load_run
from feature_selector.data import load_data
from feature_selector.evaluation import report_metrics
from feature_selector.models.autoencoder import ConvAutoencoder
from feature_selector.models.ensemble import GRUEnsembleRegressor
from feature_selector.pipeline.adapters import ModelStep
from feature_selector.pipeline.frozen import FrozenPipeline
from feature_selector.visualization import plot_split_predictions, plot_training_history

config = load_config(Path("configs/uzk/autoencoder/ensemble.yaml"))


## Обучение ансамбля с готовым энкодером

Энкодер ранее обучен по всем полным train-окнам, независимо от наличия цели.
Теперь его латентные последовательности поступают в GRU вместо исходных признаков.
Окна и scaler признаков берутся из экспорта; повторно задавать их в этом
пайплайне не нужно. Скейлер цели обучается на целях пригодных train-окон.

Число GRU задаёт `n_estimators`, параметры обучения — `gru_params`.
При `patience >= epochs` выполняются все эпохи с выбором лучших весов по validation.
Ridge обучается на всей той же validation. Запуск сохраняет весь пайплайн,
включая фиксированный энкодер; файл экспорта для последующего predict уже не нужен.


In [ ]:
run = run_experiment(config)
ensemble = run.model
assert isinstance(ensemble, GRUEnsembleRegressor)
print(f"Results: {run.artifact_dir}")
display(pd.DataFrame.from_dict(run.metrics, orient="index"))

member_summary = pd.DataFrame([
    {
        "model": record["estimator"],
        "seed": record["random_state"],
        "epochs": len(record["history"]),
        "best_epoch": record["best_epoch"],
        "train_windows": sum(
            batch["samples"] for batch in record["history"][0]["batches"]
        ),
    }
    for record in ensemble.history_
])
display(member_summary)


frozen = run.pipeline["encoder"]
assert isinstance(frozen, FrozenPipeline)
encoder_step = frozen.artifact_.pipeline.steps[-1][1]
assert isinstance(encoder_step, ModelStep)
encoder = encoder_step.estimator
assert isinstance(encoder, ConvAutoencoder)
print("Autoencoder train windows (pretraining):",
      sum(batch["samples"] for batch in encoder.history_[0]["batches"]))
print("Encoded window shape:", encoder.latent_shape_)


## История предобучения энкодера и выбранной GRU

Выберите `member_index` от 0 до `n_estimators - 1`.
История автоэнкодера загружена из экспорта: это предобучение реконструкции,
а не новые эпохи в текущем запуске. Его loss измеряется в единицах признаков;
loss GRU — в единицах масштабированной цели. Эти величины относятся к разным задачам.
Линии отмечают выбранные эпохи; истории всех GRU сохраняются в `history.json` и `run.pkl`.


In [ ]:
member_index = 0
member = ensemble.estimators_[member_index]
fig, axes = plt.subplots(2, 1, figsize=(12, 8), layout="constrained")
plot_training_history(
    encoder.history_, encoder.best_epoch_, ax=axes[0], title="Autoencoder pretraining"
)
axes[0].set_ylabel("MSE (scaled features)")
plot_training_history(
    member.history_, member.best_epoch_, ax=axes[1], title=f"GRU {member_index}"
)
axes[1].set_ylabel("MSE (scaled target)")
plt.show()


## Коэффициенты Ridge

Каждой GRU соответствует один столбец входа Ridge.
Коэффициенты не обязаны быть положительными или суммироваться в единицу.
Validation используется для обучения Ridge, поэтому её метрики описывают
качество на данных настройки ансамбля; test оценивается отдельно ниже.


In [ ]:
ridge_weights = member_summary[["model", "seed"]].copy()
ridge_weights["coefficient"] = ensemble.meta_model_.coef_
display(ridge_weights)
print("Ridge intercept (scaled target):", ensemble.meta_model_.intercept_)


## Загрузка и явная оценка test

Загруженный пайплайн содержит фиксированный энкодер, его preprocessing,
все GRU, Ridge и скейлер цели. Для predict исходный файл экспорта не требуется.
Предсказание не выполняет дообучение и не меняет автоматически сохранённые
train/validation-отчёты. Метрики рассчитаны только для наблюдавшихся целей.


In [ ]:
saved = load_run(run.artifact_dir)
data = load_data(**config.data)
test_report = saved.predict(data, split="test")
test_metrics = report_metrics(test_report)
reports = {**run.predictions, "test": test_report}
plot_split_predictions(reports, title="Encoded features → GRU ensemble → Ridge")
plt.show()
display(pd.DataFrame.from_dict({**run.metrics, "test": test_metrics}, orient="index"))


## Loss по тренировочным батчам

Показана GRU, выбранная через `member_index` выше.
Batch loss записывается до обновления весов; step растёт внутри одной модели.
Для TensorBoard используйте `uv run tensorboard --logdir runs`.
Loss и learning rate каждой GRU находятся под `pipeline/model/gru_<index>`,
итоговые метрики ансамбля — под `results/`.


In [ ]:
batch_history = pd.DataFrame([
    {"epoch": epoch["epoch"], **batch}
    for epoch in member.history_ for batch in epoch["batches"]
])
display(batch_history.head())
if not batch_history.empty:
    batch_history.plot(
        x="step", y="train_loss", title=f"GRU {member_index}: training batch loss"
    )
    plt.show()


## Test-метрики по десяти сидам

Запускаем новую серию с сидами 0–9 и текущим `config`.
Предобработка обучается в первом запуске и затем остаётся фиксированной;
меняется `random_state` конечной модели. Готовые энкодеры не переобучаются.
Для ансамбля каждый сид задаёт начало последовательности сидов его GRU.

Ниже показаны test-метрики каждого запуска и среднее ± выборочное стандартное
отклонение (`ddof=1`). MAPE выражена в процентах. `n_valid` / `n_total` показывает,
для скольких запусков метрика определена. Test не используется для выбора сида.
Метрики каждого сида, сводка, модели и прогнозы сохраняются в каталоге серии.


In [ ]:
from IPython.display import display

from feature_selector.experiment import run_seed_experiments

run_seed_experiment = False
seed_results = None

if run_seed_experiment:
    seeds = list(range(0, 100, 10))
    seed_results = run_seed_experiments(config, seeds=seeds)
    print(f"Результаты серии: {seed_results.artifact_dir}")



In [ ]:
if seed_results is not None:
    print("Test-метрики каждого сида:")
    display(seed_results.metrics_per_seed.set_index("seed"))

    print("Test-метрики: среднее ± стандартное отклонение")
    seed_summary = seed_results.metrics_summary.copy()
    seed_summary["mean ± std"] = [
        f"{mean:.6g} ± {std:.6g}"
        for mean, std in zip(seed_summary["mean"], seed_summary["std"], strict=True)
    ]
    display(seed_summary.set_index("metric")[["mean ± std", "n_valid", "n_total"]])
